# SCLC Validation Workflow Summary

**Project:** Geneformer-based Analysis of Small Cell Lung Cancer (SCLC) T-cell Dysfunction

**Date:** August 2026

**Compute Environment:** NVIDIA GB10 (DGX Spark), Geneformer V2 104M

---

## Overview

This notebook summarizes the complete SCLC validation workflow, which:
1. **Audits** data feasibility across multiple cohorts
2. **Fine-tunes** Geneformer for 3-class classification (SCLC/LUAD/Normal)
3. **Performs** in-silico perturbation screening (delete + overexpress)
4. **Validates** findings with orthogonal spatial transcriptomics data

The workflow is the SCLC-inclusive counterpart to the prior LUAD/LUSC/normal analysis.

---

## Part 1: Data Feasibility Audit

### 1.1 Audit Decision: GO with Conditions

A comprehensive feasibility audit was conducted to identify suitable datasets for SCLC validation.

**Primary Single-Cell Cohort:**
- **Source:** HTAN/CELLxGENE T-cell dataset (6fde3ad9-c2dc-4bea-bcb1-100192dd5877)
- **Scale:** 46,140 T cells from 42 donors
- **Composition:**
  - SCLC: 11,791 cells (19 donors)
  - LUAD: 29,829 cells (22 donors)
  - Normal: 4,520 cells (4 donors)

**Orthogonal Spatial Validation Cohort:**
- **Source:** GSE263196 (10x Visium)
- **Scale:** 15,774 spots across 5 SCLC samples

**Secondary Cross-Platform Cohort:**
- **Source:** OMIX002441
- **Scale:** 1,039 T cells from 11 patients

**Pre-registered 21-Gene Panel:**

| Category | Genes |
|----------|-------|
| Exhaustion | PDCD1, CTLA4, HAVCR2, LAG3, TIGIT, TOX, LAYN |
| Cytotoxicity | NKG7, GNLY, PRF1, GZMB, GZMH, IFNG |
| Progenitor/Memory | TCF7, SLAMF6, IL7R, CCR7 |
| SCLC Subtypes | ASCL1, NEUROD1, POU2F3, YAP1 |

### 1.2 Audit Evidence Summary

| Source | Scale | Strengths | Limitations | Verdict |
|--------|-------|-----------|-------------|---------|
| HTAN/CELLxGENE | 46,140 cells; 42 donors | Direct SCLC/LUAD/normal comparison, raw counts, Ensembl IDs | Only 4 normal donors; identity verification needed | **Primary discovery** |
| GSE263196 | 15,774 spots; 5 samples | Complete Visium count/coordinate bundles | No controls; no tumor masks | **Spatial validation** |
| OMIX002441 | 1,039 T cells; 11 patients | Independent platform; all markers tokenable | Low/imbalanced donor counts | **Secondary sensitivity** |
| GSE261348 | 175 AOIs; 1,738 targets | Pretreatment spatial immune panel | Outcome columns absent | **Blocked pending metadata** |

---

## Part 2: Cohort Preparation & Data Splitting

### 2.1 Donor Identity Resolution

Special considerations for donor mapping:

- **`PleuralEffusion`**: A biospecimen label mapping to exactly one participant (HTA8_2001)
- **Paired donors**: RU675, RU682, RU684 each contribute both LUAD tumor AND normal-tissue T cells from the same patient
- **Cross-disease guard**: These paired donors are pinned to single splits across both disease labels to prevent leakage

### 2.2 Donor-Disjoint Split (60/20/20)

Cells assigned at donor level with cell-count balancing:

| Disease | Split | Cells | Donors |
|---------|-------|-------|--------|
| **LUAD** | train | 17,831 | 14 |
| | eval | 5,611 | 4 |
| | test | 6,387 | 4 |
| **Normal** | train | 2,334 | 2 |
| | eval | 1,620 | 1 |
| | test | 566 | 1 |
| **SCLC** | train | 7,037 | 13 |
| | eval | 2,330 | 3 |
| | test | 2,424 | 3 |

✅ **Leakage checks passed:** Zero donors appear in multiple splits

---

## Part 3: Geneformer Fine-tuning

### 3.1 Tokenization

```python
# Parameters
tokenizer = TranscriptomeTokenizer(
    custom_attr_name_dict={
        "cell_id": "cell_id",
        "individual": "donor_id",
        "celltype": "cell_type",
        "disease": "disease",
        "split": "split",
        "length": "length"
    },
    nproc=4
)
```

### 3.2 Fine-tuning Configuration

| Parameter | Value |
|-----------|-------|
| Base model | Geneformer-V2-104M |
| Model type | Cell classifier |
| Classes | SCLC, LUAD, normal |
| Epochs | 1 |
| Learning rate | 5e-5 |
| Training batch size | 8 |
| Eval batch size | 16 |
| Frozen layers | 6 (transformer layers) |
| Random seed | 43 |

**Key Design Decision:** Training cells only used for reference centroids; held-out test cells never contribute to reference embeddings

---

## Part 4: Classification Performance

### 4.1 Overall Metrics (Held-out Test Set)

```json
{
  "overall_accuracy": 0.919,
  "overall_macro_f1": 0.903,
  "eval_macro_f1": 0.830,
  "eval_accuracy": 0.882
}
```

### 4.2 Per-Class Performance

| Disease | Precision | Recall | F1 | Test Cells | Test Donors |
|---------|-----------|--------|-----|------------|-------------|
| **LUAD** | 0.926 | 0.959 | 0.942 | 6,386 | 4 |
| **Normal** | 0.847 | 0.986 | 0.911 | 566 | 1 ⚠️ |
| **SCLC** | 0.922 | 0.800 | 0.857 | 2,424 | 3 |

⚠️ **Caveat:** Normal class has only 1 test donor - cannot distinguish real signal from single-donor idiosyncrasies

### 4.3 Context Comparison

| Model | Accuracy | Macro F1 |
|-------|----------|----------|
| **SCLC/LUAD/Normal** (this work) | 0.919 | 0.903 |
| Prior LUAD/LUSC/Normal | 0.783 | 0.758 |

*Note: Different classes/cohort - calibration reference only, not head-to-head*

---

## Part 5: In-Silico Perturbation Screening

### 5.1 Experimental Design

**Two perturbation types:**
- **Delete**: Remove gene from rank-value encoding
- **Overexpress**: Move gene to front of rank-value encoding (maximal expression)

**Efficient three-source design:**

| Source Screen | Directional Comparisons |
|---------------|------------------------|
| SCLC | SCLC → LUAD; SCLC → Normal |
| LUAD | LUAD → SCLC; LUAD → Normal |
| Normal | Normal → SCLC; Normal → LUAD |

**Total:** 6 directional comparisons × 2 perturbation types = **12 stats tables**

### 5.2 Targeted Gene Panel (50 genes)

- **21 pre-registered panel genes**: Exhaustion, cytotoxicity, progenitor, SCLC-subtype markers
- **29 top drivers**: From prior LUAD/LUSC/normal screen (contamination markers excluded)

### 5.3 Concordance Analysis

Primary evidence tier: **Deletion vs. Overexpression concordance**

```
shift_s = cosine(perturbed_cell, reference_s) - cosine(original_cell, reference_s)
```

- **Positive shift**: Movement toward goal state
- **Negative shift**: Movement away from goal state
- **Concordant hit**: Both delete and overexpress FDR < 0.05 with opposite-sign shifts

### 5.4 Headline Results: Internal Validation via SCLC Master Regulators

**ASCL1 and NEUROD1** (canonical SCLC neuroendocrine TFs) show the strongest concordant signal:

| Gene | Delete shift (toward LUAD) | Overexpress shift | Delete FDR | Overexpress FDR | N |
|------|---------------------------|-------------------|------------|-----------------|---|
| **NEUROD1** | +0.378 | -0.013 | 1.3e-7 | 4.2e-132 | 12 |
| **ASCL1** | +0.157 | -0.037 | 2.1e-30 | ~0 | 73 |

**Interpretation:**
- Deleting either gene moves SCLC cells toward LUAD (loses SCLC identity)
- Overexpressing either moves cells further toward SCLC (away from LUAD)
- This is the **expected direction** for SCLC master regulators
- Functions as internal positive control for the pipeline

⚠️ **Caveat:** Low detection (N=12, 73) - very few T cells express these tumor-intrinsic TFs

### 5.5 Panel Gene Results: Best-Powered Concordant Hits

**Summary:** 51 of 126 panel-gene × comparison combinations are concordant

**Well-powered hits (N ≥ 300 detections):**

| Comparison | Gene | Delete shift | Overexpress shift | Delete N | Interpretation |
|------------|------|--------------|-------------------|----------|----------------|
| LUAD→SCLC | TIGIT | +0.0069 | -0.0128 | 1,320 | Exhaustion marker |
| LUAD→SCLC | GZMH | +0.0057 | -0.0050 | 1,621 | Cytotoxicity |
| LUAD→Normal | GZMH | -0.0047 | +0.0056 | 1,621 | Cytotoxicity |
| LUAD→Normal | CCR7 | +0.0047 | -0.0182 | 1,561 | Progenitor/memory |
| LUAD→SCLC | CCR7 | -0.0016 | +0.0056 | 1,561 | Progenitor/memory |
| SCLC→Normal | GNLY | +0.0030 | -0.0011 | 1,183 | Cytotoxicity |
| LUAD→Normal | NKG7 | +0.0025 | -0.0091 | 2,819 | Cytotoxicity |
| LUAD→SCLC | PRF1 | +0.0016 | -0.0008 | 1,534 | Cytotoxicity |
| SCLC→Normal | IL7R | +0.0013 | -0.0054 | 1,131 | Progenitor/memory |
| LUAD→SCLC | LAG3 | -0.0008 | +0.0062 | 1,190 | Exhaustion marker |
| LUAD→Normal | LAG3 | +0.0008 | -0.0116 | 1,190 | Exhaustion marker |
| LUAD→SCLC | TCF7 | -0.0007 | +0.0066 | 1,431 | Progenitor/memory |

**Key Patterns:**
- Exhaustion markers (TIGIT, LAG3) and cytotoxicity markers (GZMH, NKG7, PRF1, GNLY) show robust effects around LUAD arm
- Progenitor/memory markers (CCR7, IL7R, TCF7) show opposite-direction effects
- Effect sizes small (0.001-0.02) but consistent with 1,000+ detections

### 5.6 Donor-Level Consistency Analysis

Each of 123 concordant hits classified by donor consistency:

| Class | N | Meaning |
|-------|---|---------|
| **Fully consistent** | 43 | 100% donors agree in sign, both arms |
| **Majority consistent** | 33 | ≥50% but <100% donors agree, both arms |
| **Inconsistent** | 9 | <50% donors agree in at least one arm |
| **Single-donor only** | 38 | <2 donors detected gene in at least one arm |

**43 fully donor-consistent hits include:**
- TIGIT (all 3 comparisons)
- GZMH, CCR7, NKG7, TCF7, IL7R, SLAMF6, CTLA4, HAVCR2, IFNG
- ASCL1 and NEUROD1 (both comparisons)

**9 inconsistent hits** (should be discounted):
- GZMB, PRF1, LAG3 (SCLC→ comparisons)
- HBA2, HBB, S100A2, MNDA, RPS27 (contamination signals)

### 5.7 Contamination Flags in Top Drivers

Several well-powered concordant hits flagged as ambient-RNA/stress candidates:

| Gene | Flag |
|------|------|
| HBA1/HBB | Red blood cell contamination (hemoglobin) |
| HSPA1B | Generic stress (heat shock) |
| RPS26 | Ribosomal, rank-abundant (N=5,583) |
| S100A8/S100A9 | Myeloid alarmins |
| TPSB2 | Mast cell tryptase |

**Recommendation:** These require biological evaluation (ambient-RNA/doublet sensitivity, T-cell subtype specificity) before treating as validated findings

---

## Part 6: Spatial Validation (GSE263196)

### 6.1 Experimental Design

**Goal:** Test whether pre-registered T-cell dysfunction signature is enriched in T-cell-rich tumor regions

**Cohort:** 5 fresh-frozen SCLC 10x Visium samples (SCLC3, SCLC4, SCLC8, SCLC9, SCLC12)

**Spot filtering:**
- In-tissue spots only (`in_tissue == 1`)
- Minimum 200 total UMI counts
- Normalized to 10,000 per spot, log1p-transformed

### 6.2 Scoring Strategy

**T-cell score:** Pan-T-cell markers (CD3D, CD3E, CD3G, CD2, CD5, CD28, TRBC1, TRBC2, IL7R, CD8A, CD8B, CD4)

**Dysfunction score:** Exhaustion markers from pre-registered panel (PDCD1, CTLA4, HAVCR2, LAG3, TIGIT, TOX, LAYN)

*Note: Different gene sets to avoid testing signature against itself*

**Analysis:** Spearman correlation per sample, pooled via inverse-variance weighted meta-analysis

### 6.3 Spatial Validation Results

| Sample | Spots | Spearman ρ | 95% CI | p-value |
|--------|-------|------------|--------|---------|
| SCLC3 | 3,849 | 0.028 | [-0.004, 0.059] | 0.086 |
| SCLC4 | 2,709 | **0.154** | [0.117, 0.190] | 8.9e-16 |
| SCLC8 | 3,030 | **0.070** | [0.035, 0.106] | 1.0e-4 |
| SCLC9 | 3,519 | **0.404** | [0.376, 0.431] | 4.1e-138 |
| SCLC12 | 2,525 | **0.116** | [0.077, 0.154] | 5.8e-9 |
| **Pooled (5 samples)** | 15,632 | **0.161** | **[0.146, 0.176]** | ~0 |

**Key Findings:**
- 4 of 5 samples show significant positive correlation individually (p < 1e-3)
- SCLC3 directionally positive but not significant alone (CI crosses zero)
- **Pooled meta-analysis: ρ = 0.161, 95% CI [0.146, 0.176]**
- Effect sizes heterogeneous (0.03-0.40) - reported as heterogeneity, not smoothed over

---

## Part 7: Key Limitations & Caveats

### 7.1 Cohort-Level

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| **Thin normal class** (4 donors: 2 train/1 eval/1 test) | Normal eval/test metrics rest on 1 donor | Named limitation; not treated as equivalent statistical power |
| **No pipeline checkpoint** | Runs into expensive perturbation regardless of classifier quality | Per explicit instruction; interpret perturbation results conditionally |
| **No normal/LUAD spatial controls** | Cannot test SCLC-specificity of spatial pattern | Acknowledged; only tests presence within SCLC |

### 7.2 Analysis-Level

| Limitation | Impact |
|------------|--------|
| **Marker-score proxy** (not deconvolution) | T-cell abundance = transcriptional signature, not true cell-type proportion |
| **Correlational** (not causal) | Positive correlation consistent with but does not establish in-situ exhaustion |
| **Between-patient heterogeneity** | Effect sizes range 0.03-0.40; pooled estimate should not read as uniform |
| **Contamination signals** | HBA/HBB, S100A8/9, ribosomal genes flagged as ambient-RNA candidates |

### 7.3 Uncompleted Steps

- Ambient-RNA / contamination correction
- Sensitivity analysis excluding ribosomal/rank-dominant genes
- Pathway-level interpretation

---

## Part 8: Summary & Conclusions

### 8.1 Workflow Completion Status

| Component | Status | Key Output |
|-----------|--------|------------|
| ✅ Data Audit | Complete | GO decision with conditions; HTAN primary, GSE263196 spatial |
| ✅ Cohort Prep | Complete | 46,140 cells; donor-disjoint splits; zero leakage |
| ✅ Fine-tuning | Complete | 91.9% accuracy, 90.3% macro F1 on test set |
| ✅ Perturbation Screen | Complete | 300 gene-runs; 123 concordant hits; 43 fully donor-consistent |
| ✅ Spatial Validation | Complete | ρ = 0.161 [0.146, 0.176] pooled; 4/5 samples significant |

### 8.2 Key Findings

1. **Internal validation**: ASCL1/NEUROD1 show expected master regulator behavior (strong concordant signal)

2. **Panel genes**: Exhaustion (TIGIT, LAG3) and cytotoxicity (GZMH, NKG7, PRF1) markers show robust concordant effects around LUAD arm

3. **Progenitor axis**: CCR7, IL7R, TCF7 show opposite-direction effects, consistent with progenitor-vs-exhausted axis

4. **Spatial validation**: T-cell abundance positively correlated with dysfunction score in SCLC tumor tissue (ρ = 0.161)

5. **Donor robustness**: 43 hits fully consistent across all donors with data

### 8.3 Claim Boundaries

**Supported:**
- SCLC/LUAD/Normal classification with held-out donor validation
- In-silico perturbation candidate genes with concordance/donor-consistency
- Spatial correlation between T-cell abundance and dysfunction in SCLC tissue

**Not yet validated (preliminary):**
- "Progressive transitions" between states
- "Terminal exhaustion-like regions"
- Specific "candidate regulators" (require biological evaluation)

---

**End of Summary**

---

## Appendix: File Locations

### Code & Scripts

```
sclc_validation/
├── audit/
│   ├── audit_sclc_data.py          # Feasibility audit
│   └── notebooks/                   # Executable audit notebook
├── perturbation_workflow/
│   ├── METHODS.md                   # Full methodology
│   ├── targeted_panel/
│   │   ├── run_targeted_panel.py   # Perturbation execution
│   │   ├── analyze_targeted_results.py  # Results analysis
│   │   └── donor_consistency.py    # Donor-level validation
│   └── notebooks/                   # Full workflow notebook
└── spatial_validation/
    ├── spatial_validation.py       # Spatial analysis
    └── plot_spatial_tissue_panel.py # Figure generation
```

### Results

```
sclc_validation/
├── audit/results/                   # Audit tables and summaries
├── perturbation_workflow/results/   # Classification metrics
├── perturbation_workflow/targeted_panel/results/  # Perturbation stats
└── spatial_validation/results/      # Spatial correlation tables
    └── figures/                     # Tissue panel and forest plots
```